Trains a chess resnet

In [1]:
# Colab Setup: clone repo and install requirements
import os, sys

# check if running in Colab
IN_COLAB = "google.colab" in sys.modules
print(f"Running on Colab:{IN_COLAB}")

if IN_COLAB:
    if not os.path.exists("vis10n"):
        print("Cloning full repo")
        !git clone https://github.com/simchaorgel/vis10n.git
    !cd vis10n && git pull
    sys.path.append("vis10n")
    # Download requirements
    print("Downloading requirements")
    !pip install -q -r vis10n/requirements.txt
else:
    # Running locally from training/: make the repo root importable
    sys.path.append("..")

Running on Colab:True
Already up to date.


In [ ]:
# Main imports
import torch as t
from dataclasses import dataclass
import torch.nn.functional as F
from tqdm.notebook import tqdm
import wandb

In [3]:
# Local imports: this cell imports the necessary files
# either locally or from colabs repo clone
from models.chess_resnet import ChessResnet
import optimizers
from dataloaders.lichess import get_lichess

In [4]:
# Check if we have access to a GPU
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print(device)

cuda


In [5]:
# Wandb login
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: simchaorgel (simchao) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# Load data
def colab_get_lichess(trainset_size: int, testset_size: int):
    """On Colab, copies the processed data from Drive into the repo's data folder first"""
    if IN_COLAB:
        processed_dir = "/content/vis10n/data/chess/processed"
        # Only copy once per session (cell may be re-run)
        if not os.path.exists(f"{processed_dir}/boards.npy"):
            from google.colab import drive
            drive.mount('/content/drive')
            !mkdir -p {processed_dir}
            !cp /content/drive/MyDrive/data/chess/processed/*.npy {processed_dir}/

    return get_lichess(trainset_size, testset_size)


def expand(boards: t.Tensor) -> t.Tensor:
    """
    (B, 8, 8) uint8 piece numbers -> (B, 12, 8, 8) float planes, as the model expects.
    Piece number 0 is an empty square, so its one_hot column is dropped
    """
    return F.one_hot(boards.long(), 13)[..., 1:].permute(0, 3, 1, 2).float()


def gpu_batches(boards: t.Tensor, moves: t.Tensor, batch_size: int, shuffle: bool):
    """
    Yields (planes, moves) batches sliced straight from tensors already on the GPU.
    Much faster than a DataLoader, which fetches and stacks every position one at a time on the CPU
    """
    n = len(boards)
    order = t.randperm(n, device=boards.device) if shuffle else t.arange(n, device=boards.device)
    for start in range(0, n, batch_size):
        idx = order[start:start + batch_size]
        yield expand(boards[idx]), moves[idx]

In [ ]:
# Training args
#####################################################
@dataclass
class ChessResnetTrainingArgs:
    trainset_size: int = 10_000_000   # more than there is: takes everything before the test set
    testset_size: int = 300_000
    batch_size: int = 512
    epochs: int = 5
    optimizer: str = "AdamW"
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    log_every: int = 20               # batches between loss logs (each log waits for the GPU)
    save_model: bool = True

In [ ]:
# Train
#####################################################
def train(args: ChessResnetTrainingArgs) -> ChessResnet:
    """
    Trains the chess ResNet on Lichess positions, logging to wandb. Returns the trained model, also saves to wandb
    """
    try:
        # Wandb init
        wandb.init(entity="simchao", project="chess-resnet", config=args)
        # Create ResNet instance
        model = ChessResnet().to(device)
        #wandb.watch(model, log="all", log_freq=50)
        # Get data
        trainset, testset = colab_get_lichess(args.trainset_size, args.testset_size)
        # The whole dataset goes on the GPU once (uint8 boards are small); batches are sliced from it there
        train_boards, train_moves = (x.to(device) for x in trainset.tensors)
        test_boards, test_moves = (x.to(device) for x in testset.tensors)
        batches_per_epoch = -(-len(train_boards) // args.batch_size)  # ceil
        # Define optimizer
        optimizer = optimizers.get_optimizer(args.optimizer)(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
        # Logging lists
        examples_seen = 0

        # For every epoch:
        for epoch in range(args.epochs):
            epoch_accurate_predictions = 0 # For validating
            pbar = tqdm(gpu_batches(train_boards, train_moves, args.batch_size, shuffle=True), total=batches_per_epoch)

            # Train - loop for every batch in this epoch
            model.train()
            for batch, (boards, moves) in enumerate(pbar):
                # Forward pass (batches are already on the GPU, expanded to planes)
                logits = model(boards)
                # Compute loss, perform backward pass
                loss = F.cross_entropy(logits, moves)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

                # Update logs & progress bar
                examples_seen += boards.shape[0]
                if batch % args.log_every == 0:
                    wandb.log({"loss": loss.item()}, step=examples_seen)
                    pbar.set_postfix(epoch=f"{epoch + 1}/{args.epochs}", loss=f"{loss:.3f}")

            # Test to get accuracy
            model.eval()
            for boards, moves in gpu_batches(test_boards, test_moves, args.batch_size, shuffle=False):
                with t.inference_mode():
                    logits = model(boards)

                # Calculate accuracy
                # Get max logit, which is treated as the models guess
                predictions = t.argmax(logits, dim=1)
                # Create tensor with 1 if the prediction was correct, 0 if not, for every img
                accuracy_bools = predictions == moves
                # Number of accurate guesses
                epoch_accurate_predictions += accuracy_bools.sum().item()

            # Before completing the epoch
            # Add accuracy percentage to list
            epoch_accuracy = epoch_accurate_predictions / len(test_boards)
            wandb.log({"accuracy": epoch_accuracy}, step=examples_seen)

        if args.save_model:
            t.save(model.state_dict(), "chess_resnet.pt")
            artifact = wandb.Artifact("chess_resnet", type="model")
            artifact.add_file("chess_resnet.pt")
            wandb.log_artifact(artifact)
            
        wandb.finish()
        return model
    
    except Exception:
        wandb.finish()
        raise

In [9]:
# Run a standard train
###################################################
args = ChessResnetTrainingArgs()
model = train(args)

Mounted at /content/drive


  0%|          | 0/1746 [00:00<?, ?it/s]

  0%|          | 0/1746 [00:00<?, ?it/s]

  0%|          | 0/1746 [00:00<?, ?it/s]

  0%|          | 0/1746 [00:00<?, ?it/s]

  0%|          | 0/1746 [00:00<?, ?it/s]

accuracy,▁▆█▇█
loss,█▇▇▆▅▅▄▄▄▄▄▄▄▄▄▄▃▂▃▃▃▃▃▃▃▃▁▂▂▂▂▂▁▁▁▂▁▂▂▂
accuracy,0.34267
loss,1.6778
